# 04 — Train Models

**Goal:** Train 7 ML models on each of the 3 poverty thresholds (21 models total).  
**Reads:** `data/processed/train.csv`, `data/processed/test.csv`  
**Outputs:** 21 `.pkl` files in `models/`

Models: XGBoost (CPU), XGBoost (GPU), LightGBM, Random Forest, Ridge, MLP, GAM  
Targets: `poverty_3`, `poverty_8_30`, `poverty_10`

> `poverty_4_20` excluded — source file contains poverty gap, not headcount ratio.

## Imports and paths

In [ ]:
import pandas as pd
import numpy as np
import time
import joblib
from pathlib import Path

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from pygam import LinearGAM, s

import warnings
warnings.filterwarnings('ignore')

DATA_PROCESSED = Path("../data/processed")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = ["gdp_per_capita", "hdi", "control_of_corruption", "employment_agriculture", "gini"]
TARGETS = ["poverty_3", "poverty_8_30", "poverty_10"]  # poverty_4_20 excluded (poverty gap data)

print(f"Features: {FEATURES}")
print(f"Targets:  {TARGETS}")

## Load data

In [ ]:
train_full = pd.read_csv(DATA_PROCESSED / "train.csv")
test_full = pd.read_csv(DATA_PROCESSED / "test.csv")

print(f"Train: {train_full.shape}")
print(f"Test:  {test_full.shape}")

## Define model configurations

No hyperparameter tuning — we use reasonable defaults informed by common practice.

In [ ]:
def get_models():
    """Return a dict of model name → model instance (fresh for each target)."""
    return {
        "XGBoost_CPU": XGBRegressor(
            n_estimators=300, max_depth=6, learning_rate=0.1,
            tree_method="hist", device="cpu", random_state=42, verbosity=0
        ),
        "XGBoost_GPU": XGBRegressor(
            n_estimators=300, max_depth=6, learning_rate=0.1,
            tree_method="hist", device="cuda", random_state=42, verbosity=0
        ),
        "LightGBM": LGBMRegressor(
            n_estimators=300, max_depth=6, learning_rate=0.1,
            random_state=42, verbosity=-1
        ),
        "RandomForest": RandomForestRegressor(
            n_estimators=300, max_depth=15, random_state=42, n_jobs=-1
        ),
        "Ridge": Ridge(alpha=1.0),
        "MLP": MLPRegressor(
            hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42,
            early_stopping=True, validation_fraction=0.1
        ),
    }

# Models that need scaled features
NEEDS_SCALING = {"Ridge", "MLP"}

print(f"Models: {list(get_models().keys())} + GAM")
print(f"Need scaling: {NEEDS_SCALING}")

## Training loop

For each poverty threshold × each model:
1. Filter rows where the target is not NaN
2. Scale features for Ridge/MLP (fit scaler on train only)
3. Fit the model
4. Save to `models/{model_name}_{threshold}.pkl`
5. For Ridge/MLP, also save the scaler

In [ ]:
# Track training times
timing_records = []

for target in TARGETS:
    print("=" * 60)
    print(f"TARGET: {target}")
    print("=" * 60)
    
    # Filter rows where this target is not NaN
    train_df = train_full.dropna(subset=[target])
    test_df = test_full.dropna(subset=[target])
    
    X_train = train_df[FEATURES].values
    y_train = train_df[target].values
    X_test = test_df[FEATURES].values
    y_test = test_df[target].values
    
    print(f"  Train rows: {len(X_train)}, Test rows: {len(X_test)}")
    
    # Fit scaler on train data (for Ridge/MLP)
    scaler = StandardScaler().fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Save scaler for this target
    joblib.dump(scaler, MODELS_DIR / f"scaler_{target}.pkl")
    
    # --- Train sklearn/xgboost/lightgbm models ---
    models = get_models()
    for model_name, model in models.items():
        
        # Choose scaled or unscaled features
        X_tr = X_train_scaled if model_name in NEEDS_SCALING else X_train
        
        # GPU fallback for XGBoost
        if model_name == "XGBoost_GPU":
            try:
                t0 = time.time()
                model.fit(X_tr, y_train)
                elapsed = time.time() - t0
            except Exception as e:
                print(f"  {model_name}: GPU failed ({e}), falling back to CPU")
                model = XGBRegressor(
                    n_estimators=300, max_depth=6, learning_rate=0.1,
                    tree_method="hist", device="cpu", random_state=42, verbosity=0
                )
                t0 = time.time()
                model.fit(X_tr, y_train)
                elapsed = time.time() - t0
        else:
            t0 = time.time()
            model.fit(X_tr, y_train)
            elapsed = time.time() - t0
        
        # Save model
        save_path = MODELS_DIR / f"{model_name}_{target}.pkl"
        joblib.dump(model, save_path)
        
        timing_records.append({
            "target": target, "model": model_name, "time_sec": round(elapsed, 3)
        })
        print(f"  {model_name:20s} — {elapsed:.3f}s — saved to {save_path.name}")
    
    # --- Train GAM separately (different API) ---
    t0 = time.time()
    gam = LinearGAM(s(0) + s(1) + s(2) + s(3) + s(4))
    gam.fit(X_train, y_train)
    elapsed = time.time() - t0
    
    save_path = MODELS_DIR / f"GAM_{target}.pkl"
    joblib.dump(gam, save_path)
    
    timing_records.append({
        "target": target, "model": "GAM", "time_sec": round(elapsed, 3)
    })
    print(f"  {'GAM':20s} — {elapsed:.3f}s — saved to {save_path.name}")
    print()

## Training time summary

In [ ]:
timing_df = pd.DataFrame(timing_records)

# Pivot: rows = model, columns = target
timing_pivot = timing_df.pivot(index="model", columns="target", values="time_sec")
timing_pivot["total"] = timing_pivot.sum(axis=1)
timing_pivot = timing_pivot.sort_values("total")

print("Training time (seconds):")
print(timing_pivot.round(3))
print(f"\nTotal training time: {timing_df['time_sec'].sum():.1f}s")

## Verify saved models

In [ ]:
saved_models = sorted(MODELS_DIR.glob("*.pkl"))
print(f"Total files saved: {len(saved_models)}")
for p in saved_models:
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:45s} {size_kb:8.1f} KB")

---
**Outputs produced:** 21 model files + 3 scaler files in `models/`  
- 7 models × 3 poverty thresholds = 21 `.pkl` files  
- 3 `scaler_{target}.pkl` files (for Ridge/MLP at prediction time)  

> `poverty_4_20` excluded — source file contains poverty gap, not headcount ratio.

Next: notebook 05 evaluates all models on the test set.